# New Relic Service Map — Tests

Simple smoke/unit tests for `newrelic_service_map.py`.  
Cells 1–3 use `unittest.mock` so no real API key is needed.  
Cell 4 is an optional live smoke test — set `NR_API_KEY` in your environment to run it.

In [ ]:
# Cell 1 — Setup: add parent dir to path and import module
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import patch, MagicMock
import newrelic_service_map as nrsm

FAKE_KEY = "FAKE_API_KEY_FOR_TESTS"
print("Import OK")

In [ ]:
# Cell 2 — Test search_entities_by_tag returns a list of entity dicts

MOCK_SEARCH_RESPONSE = {
    "data": {
        "actor": {
            "entitySearch": {
                "results": {
                    "entities": [
                        {
                            "name": "payments-service",
                            "guid": "GUID_1",
                            "entityType": "APM_APPLICATION_ENTITY",
                            "tags": [{"key": "environment", "values": ["production"]}],
                        },
                        {
                            "name": "orders-service",
                            "guid": "GUID_2",
                            "entityType": "APM_APPLICATION_ENTITY",
                            "tags": [{"key": "environment", "values": ["production"]}],
                        },
                    ]
                }
            }
        }
    }
}

mock_resp = MagicMock()
mock_resp.json.return_value = MOCK_SEARCH_RESPONSE
mock_resp.raise_for_status = MagicMock()

with patch("requests.post", return_value=mock_resp) as mock_post:
    entities = nrsm.search_entities_by_tag("environment", "production", FAKE_KEY)

assert isinstance(entities, list), "Expected a list"
assert len(entities) == 2, f"Expected 2 entities, got {len(entities)}"
assert entities[0]["name"] == "payments-service"
assert entities[0]["tags"] == {"environment": ["production"]}
assert mock_post.call_args[1]["headers"]["Api-Key"] == FAKE_KEY
print("PASS: search_entities_by_tag returned correct structure")

In [ ]:
# Cell 3 — Test extract_service_map returns expected graph shape

MOCK_REL_RESPONSE = {
    "data": {
        "actor": {
            "entity": {
                "name": "payments-service",
                "guid": "GUID_1",
                "entityType": "APM_APPLICATION_ENTITY",
                "tags": [],
                "relatedEntities": {
                    "results": [
                        {
                            "source": {"entity": {"name": "payments-service", "guid": "GUID_1", "entityType": "APM_APPLICATION_ENTITY"}},
                            "target": {"entity": {"name": "db-postgres", "guid": "GUID_DB", "entityType": "GENERIC_INFRASTRUCTURE_ENTITY"}},
                            "type": "CALLS",
                        }
                    ]
                },
            }
        }
    }
}

# Alternate mock for orders-service (no outbound calls)
MOCK_REL_RESPONSE_2 = {
    "data": {
        "actor": {
            "entity": {
                "name": "orders-service",
                "guid": "GUID_2",
                "entityType": "APM_APPLICATION_ENTITY",
                "tags": [],
                "relatedEntities": {"results": []},
            }
        }
    }
}

call_count = 0

def side_effect(*args, **kwargs):
    global call_count
    r = MagicMock()
    r.raise_for_status = MagicMock()
    # First call: entitySearch, next two: get_entity_relationships per entity
    if call_count == 0:
        r.json.return_value = MOCK_SEARCH_RESPONSE
    elif call_count == 1:
        r.json.return_value = MOCK_REL_RESPONSE
    else:
        r.json.return_value = MOCK_REL_RESPONSE_2
    call_count += 1
    return r

with patch("requests.post", side_effect=side_effect):
    graph = nrsm.extract_service_map("environment", "production", FAKE_KEY)

assert "entities" in graph, "graph must have 'entities' key"
assert "edges" in graph, "graph must have 'edges' key"
assert "GUID_1" in graph["entities"], "GUID_1 should be in entities"
assert "GUID_2" in graph["entities"], "GUID_2 should be in entities"
assert "GUID_DB" in graph["entities"], "Related entity GUID_DB should be added"
assert len(graph["edges"]) == 1, f"Expected 1 edge, got {len(graph['edges'])}"
assert graph["edges"][0]["source_guid"] == "GUID_1"
assert graph["edges"][0]["target_guid"] == "GUID_DB"
print("PASS: extract_service_map returns correct graph structure")
print(f"  Entities: {list(graph['entities'].keys())}")
print(f"  Edges: {graph['edges']}")

In [ ]:
# Cell 4 — Optional live smoke test (requires NR_API_KEY env var)

import os

# Optionally load from a .env file in the project root
try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join("..", ".env"))
except ImportError:
    pass

api_key = os.environ.get("NR_API_KEY")

if not api_key:
    print("SKIP: NR_API_KEY not set — skipping live smoke test")
else:
    tag_key = os.environ.get("NR_TAG_KEY", "environment")
    tag_value = os.environ.get("NR_TAG_VALUE", "production")
    print(f"Running live search for tags.{tag_key} = '{tag_value}' ...")
    entities = nrsm.search_entities_by_tag(tag_key, tag_value, api_key)
    print(f"Found {len(entities)} entities")
    for e in entities[:5]:
        print(f"  - {e['name']} ({e['entityType']}) [{e['guid']}]")
    if entities:
        print("\nFetching relationships for first entity...")
        detail = nrsm.get_entity_relationships(entities[0]["guid"], api_key)
        print(f"  Relationships: {len(detail.get('relationships', []))}")
    print("PASS: live smoke test completed")